In [1]:
pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
import copy
import math
import random
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import lpips


# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    use_amp: bool = True

    dataset_root: str = "/kaggle/input/datasets/takihasan/div2k-dataset-for-super-resolution/Dataset"
    train_hr_dir: str = "/kaggle/input/datasets/takihasan/div2k-dataset-for-super-resolution/Dataset/DIV2K_train_HR"
    train_lr_dir: str = "/kaggle/input/datasets/takihasan/div2k-dataset-for-super-resolution/Dataset/DIV2K_train_LR_bicubic_X4/X4"
    val_hr_dir: str = "/kaggle/input/datasets/takihasan/div2k-dataset-for-super-resolution/Dataset/DIV2K_valid_HR"
    val_lr_dir: str = "/kaggle/input/datasets/takihasan/div2k-dataset-for-super-resolution/Dataset/DIV2K_valid_LR_bicubic_X4/X4"

    scale: int = 4
    patch_size: int = 96
    batch_size: int = 4
    val_batch_size: int = 1
    num_workers: int = 2

    in_channels: int = 3
    noise_image_channels: int = 3
    noise_embed_dim: int = 256
    unet_base: int = 64
    unet_channel_mult: Tuple[int, ...] = (1, 2, 4)
    unet_num_blocks: int = 2
    residual_out_scale: float = 1.0

    epochs: int = 40
    lr: float = 2e-4
    min_lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    ema_decay: float = 0.999

    # Drifting setup
    num_samples_per_lr: int = 6
    num_positive_views: int = 6
    temperature: float = 0.10

    # softened drift strength
    drift_step_size: float = 0.03
    drift_loss_weight: float = 0.10
    drift_loss_weight_max: float = 0.25
    warmup_epochs: int = 10
    drift_ramp_epochs: int = 8

    lambda_pos: float = 1.0
    lambda_same_neg_start: float = 0.20
    lambda_same_neg_end: float = 0.35

    feature_max_positions: int = 32
    feature_scale_index: int = 2   # one VGG scale only

    # normalization
    norm_eps: float = 1e-6
    feat_scale_min: float = 1e-4
    drift_fp32: bool = True

    # soft drift normalization:
    # small drift is not amplified; only large drift is mildly normalized
    drift_scale_center: float = 1.0
    drift_scale_max: float = 2.0

    pixel_l1_w: float = 0.1
    lr_consistency_w: float = 1.0

    # richer positives
    positive_sharpen_amount: Tuple[float, float] = (0.03, 0.07)
    positive_sharpen_sigma: Tuple[float, float] = (0.7, 1.2)
    positive_highpass_amount: Tuple[float, float] = (0.02, 0.05)
    positive_combo_highpass_amount: Tuple[float, float] = (0.015, 0.035)
    positive_lr_tolerance: float = 0.02

    log_every: int = 50
    eval_every_epochs: int = 1
    val_batches: int = 20
    save_every_epochs: int = 5

    eval_use_zero_noise: bool = True
    eval_on_y_channel: bool = True
    shave_border: int = 4
    eval_lpips_net: str = "vgg"
    eval_lpips_shave_border: int = 4

    save_dir: str = "/kaggle/working/drifting_sr_singlelevel_batchfeat_softdrift"
    ckpt_dir: str = "/kaggle/working/drifting_sr_singlelevel_batchfeat_softdrift/checkpoints"
    history_dir: str = "/kaggle/working/drifting_sr_singlelevel_batchfeat_softdrift/histories"
    plot_dir: str = "/kaggle/working/drifting_sr_singlelevel_batchfeat_softdrift/plots"


cfg = Config()


# ============================================================
# Repro / dirs
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: Config):
    os.makedirs(cfg.save_dir, exist_ok=True)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    os.makedirs(cfg.history_dir, exist_ok=True)
    os.makedirs(cfg.plot_dir, exist_ok=True)


# ============================================================
# Dataset
# ============================================================
class DIV2KPairDataset(Dataset):
    def __init__(self, hr_dir, lr_dir, scale=4, patch_size=None, training=True):
        self.hr_dir = Path(hr_dir)
        self.lr_dir = Path(lr_dir)
        self.scale = scale
        self.patch_size = patch_size
        self.training = training
        self.to_tensor = transforms.ToTensor()

        self.hr_files = sorted(self.hr_dir.glob("*.png"))
        if len(self.hr_files) == 0:
            raise RuntimeError(f"No HR png files found in: {self.hr_dir}")
        if self.training and (self.patch_size is None or self.patch_size % self.scale != 0):
            raise ValueError("patch_size must be set and divisible by scale for training")

    def __len__(self):
        return len(self.hr_files)

    def _lr_path_from_hr(self, hr_path: Path) -> Path:
        return self.lr_dir / f"{hr_path.stem}x{self.scale}.png"

    def _random_crop_pair(self, hr: Image.Image, lr: Image.Image):
        hr_ps = self.patch_size
        lr_ps = hr_ps // self.scale
        lr_w, lr_h = lr.size
        x_lr = random.randint(0, lr_w - lr_ps)
        y_lr = random.randint(0, lr_h - lr_ps)
        x_hr = x_lr * self.scale
        y_hr = y_lr * self.scale
        return (
            hr.crop((x_hr, y_hr, x_hr + hr_ps, y_hr + hr_ps)),
            lr.crop((x_lr, y_lr, x_lr + lr_ps, y_lr + lr_ps)),
        )

    def _augment_pair(self, hr: Image.Image, lr: Image.Image):
        if random.random() < 0.5:
            hr = hr.transpose(Image.FLIP_LEFT_RIGHT)
            lr = lr.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.5:
            hr = hr.transpose(Image.FLIP_TOP_BOTTOM)
            lr = lr.transpose(Image.FLIP_TOP_BOTTOM)
        if random.random() < 0.5:
            hr = hr.transpose(Image.ROTATE_90)
            lr = lr.transpose(Image.ROTATE_90)
        return hr, lr

    def _modcrop_pair(self, hr: Image.Image, lr: Image.Image):
        lr_w, lr_h = lr.size
        hr_w, hr_h = hr.size
        lr_w = min(lr_w, hr_w // self.scale)
        lr_h = min(lr_h, hr_h // self.scale)
        hr_w = lr_w * self.scale
        hr_h = lr_h * self.scale
        return hr.crop((0, 0, hr_w, hr_h)), lr.crop((0, 0, lr_w, lr_h))

    def __getitem__(self, idx):
        hr_path = self.hr_files[idx]
        lr_path = self._lr_path_from_hr(hr_path)

        hr = Image.open(hr_path).convert("RGB")
        lr = Image.open(lr_path).convert("RGB")

        if self.training:
            hr, lr = self._random_crop_pair(hr, lr)
            hr, lr = self._augment_pair(hr, lr)
        else:
            hr, lr = self._modcrop_pair(hr, lr)

        return self.to_tensor(hr), self.to_tensor(lr)


# ============================================================
# Image ops
# ============================================================
def upsample_lr(lr: torch.Tensor, scale: int = 4, out_hw: Optional[Tuple[int, int]] = None) -> torch.Tensor:
    if out_hw is not None:
        return F.interpolate(lr, size=out_hw, mode="bicubic", align_corners=False)
    return F.interpolate(lr, scale_factor=scale, mode="bicubic", align_corners=False)


def degrade_to_lr(x_hr: torch.Tensor, scale=4) -> torch.Tensor:
    h, w = x_hr.shape[-2:]
    return F.interpolate(x_hr, size=(h // scale, w // scale), mode="bicubic", align_corners=False)


def gaussian_kernel(size=5, sigma=1.0, channels=3, device="cpu", dtype=torch.float32):
    coords = torch.arange(size, device=device, dtype=dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma * sigma))
    g = g / g.sum()
    kernel2d = torch.outer(g, g)
    kernel2d = kernel2d / kernel2d.sum()
    return kernel2d.view(1, 1, size, size).repeat(channels, 1, 1, 1)


def blur_image(x: torch.Tensor, kernel_size=5, sigma=1.0) -> torch.Tensor:
    kernel = gaussian_kernel(kernel_size, sigma, channels=x.shape[1], device=x.device, dtype=x.dtype)
    pad = kernel_size // 2
    return F.conv2d(x, kernel, padding=pad, groups=x.shape[1])


def unsharp_mask(x: torch.Tensor, amount=0.05, sigma=1.0) -> torch.Tensor:
    blur = blur_image(x, kernel_size=5, sigma=sigma)
    return (x + amount * (x - blur)).clamp(0.0, 1.0)


def mild_highpass_boost(x: torch.Tensor, amount=0.03) -> torch.Tensor:
    low = blur_image(x, kernel_size=5, sigma=1.0)
    hp = x - low
    return (x + amount * hp).clamp(0.0, 1.0)


def enforce_lr_consistency(candidate: torch.Tensor, fallback: torch.Tensor, lr: torch.Tensor, cfg: Config) -> torch.Tensor:
    lr_cand = degrade_to_lr(candidate, scale=cfg.scale)
    keep = (lr_cand - lr).abs().mean(dim=(1, 2, 3), keepdim=True) <= cfg.positive_lr_tolerance
    return torch.where(keep, candidate, fallback)


def build_positive_bank(hr: torch.Tensor, lr: torch.Tensor, cfg: Config) -> torch.Tensor:
    """
    Returns exactly cfg.num_positive_views positives:
      1) GT
      2) unsharp
      3) highpass
      4) unsharp + highpass
    If num_positive_views > 4, the pattern cycles with fresh random params.
    """
    positives = [hr]

    builders = [
        lambda x: unsharp_mask(
            x,
            amount=random.uniform(*cfg.positive_sharpen_amount),
            sigma=random.uniform(*cfg.positive_sharpen_sigma),
        ),
        lambda x: mild_highpass_boost(
            x,
            amount=random.uniform(*cfg.positive_highpass_amount),
        ),
        lambda x: mild_highpass_boost(
            unsharp_mask(
                x,
                amount=random.uniform(*cfg.positive_sharpen_amount),
                sigma=random.uniform(*cfg.positive_sharpen_sigma),
            ),
            amount=random.uniform(*cfg.positive_combo_highpass_amount),
        ),
    ]

    while len(positives) < cfg.num_positive_views:
        builder = builders[(len(positives) - 1) % len(builders)]
        cand = builder(hr)
        cand = enforce_lr_consistency(cand, hr, lr, cfg)
        positives.append(cand)

    return torch.stack(positives[:cfg.num_positive_views], dim=1)


# ============================================================
# Metrics
# ============================================================
def rgb_to_y_channel_torch(x: torch.Tensor) -> torch.Tensor:
    r = x[:, 0:1]
    g = x[:, 1:2]
    b = x[:, 2:3]
    return 0.299 * r + 0.587 * g + 0.114 * b


def shave_tensor(x: torch.Tensor, border: int) -> torch.Tensor:
    if border <= 0:
        return x
    h, w = x.shape[-2:]
    if h <= 2 * border or w <= 2 * border:
        return x
    return x[..., border:h - border, border:w - border]


@torch.no_grad()
def calc_psnr_sr(x, y, shave=4, use_y=True):
    if use_y:
        x = rgb_to_y_channel_torch(x)
        y = rgb_to_y_channel_torch(y)
    x = shave_tensor(x, shave)
    y = shave_tensor(y, shave)
    mse = F.mse_loss(x, y)
    return (-10.0 * torch.log10(mse + 1e-12)).item()


@torch.no_grad()
def calc_lpips_sr(x, y, lpips_model, shave=4):
    x = shave_tensor(x.clamp(0.0, 1.0), shave)
    y = shave_tensor(y.clamp(0.0, 1.0), shave)
    x = x * 2.0 - 1.0
    y = y * 2.0 - 1.0
    return float(lpips_model(x, y).mean().item())


# ============================================================
# Generator
# ============================================================
class ZeroConv2d(nn.Conv2d):
    def reset_parameters(self):
        nn.init.zeros_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)


class NoiseMLP(nn.Module):
    def __init__(self, noise_embed_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, z):
        return self.net(z)


class AdaGNResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, cond_dim: int, up=False, down=False):
        super().__init__()
        self.up = up
        self.down = down

        self.norm1 = nn.GroupNorm(min(32, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.norm2 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.cond_proj = nn.Linear(cond_dim, 4 * out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)
        self.act = nn.SiLU()

    def _resample(self, x):
        if self.up:
            x = F.interpolate(x, scale_factor=2.0, mode="bilinear", align_corners=False)
        if self.down:
            x = F.avg_pool2d(x, kernel_size=2, stride=2)
        return x

    def forward(self, x, cond):
        h = self._resample(x)
        x_skip = self._resample(x)

        h = self.conv1(self.act(self.norm1(h)))

        scale1, shift1, scale2, shift2 = self.cond_proj(cond).chunk(4, dim=1)

        h = self.norm2(h)
        h = h * (1 + scale1[:, :, None, None]) + shift1[:, :, None, None]
        h = self.conv2(self.act(h))
        h = h * (1 + scale2[:, :, None, None]) + shift2[:, :, None, None]

        return h + self.skip(x_skip)


class NoiseConditionalResidualUNetSR(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        noise_image_channels: int = 3,
        base: int = 64,
        channel_mult: Tuple[int, ...] = (1, 2, 4),
        num_blocks: int = 2,
        noise_embed_dim: int = 256,
        residual_out_scale: float = 1.0,
        scale: int = 4,
    ):
        super().__init__()
        self.noise_image_channels = noise_image_channels
        self.noise_embed_dim = noise_embed_dim
        self.residual_out_scale = residual_out_scale
        self.scale = scale

        self.cond_mlp = NoiseMLP(noise_embed_dim, base * 4)
        cond_dim = base * 4
        self.in_conv = nn.Conv2d(in_channels + noise_image_channels, base, 3, padding=1)

        dims = [base * m for m in channel_mult]
        self.downs = nn.ModuleList()
        cur_ch = base
        skip_channels = []

        for level, ch in enumerate(dims):
            if level == 0:
                for _ in range(num_blocks):
                    block = AdaGNResBlock(cur_ch, ch, cond_dim)
                    self.downs.append(block)
                    cur_ch = ch
                    skip_channels.append(cur_ch)
            else:
                block = AdaGNResBlock(cur_ch, ch, cond_dim, down=True)
                self.downs.append(block)
                cur_ch = ch
                skip_channels.append(cur_ch)
                for _ in range(num_blocks - 1):
                    block = AdaGNResBlock(cur_ch, ch, cond_dim)
                    self.downs.append(block)
                    cur_ch = ch
                    skip_channels.append(cur_ch)

        self.mid1 = AdaGNResBlock(cur_ch, cur_ch, cond_dim)
        self.mid2 = AdaGNResBlock(cur_ch, cur_ch, cond_dim)

        self.ups = nn.ModuleList()
        rev_dims = list(reversed(dims))
        rev_skips = list(reversed(skip_channels))

        for level, ch in enumerate(rev_dims):
            for block_idx in range(num_blocks):
                skip_ch = rev_skips.pop(0)
                in_ch = cur_ch + skip_ch
                up_flag = (block_idx == 0 and level > 0)
                self.ups.append(AdaGNResBlock(in_ch, ch, cond_dim, up=up_flag))
                cur_ch = ch

        self.out_norm = nn.GroupNorm(min(32, cur_ch), cur_ch)
        self.out_act = nn.SiLU()
        self.out_conv = ZeroConv2d(cur_ch, in_channels, 3, padding=1)

    def forward(
        self,
        lr: torch.Tensor,
        noise_img: Optional[torch.Tensor] = None,
        noise_vec: Optional[torch.Tensor] = None,
    ):
        b, _, h_lr, w_lr = lr.shape
        H, W = h_lr * self.scale, w_lr * self.scale

        x_up = upsample_lr(lr, out_hw=(H, W), scale=self.scale)

        if noise_img is None:
            noise_img = torch.randn(b, self.noise_image_channels, H, W, device=lr.device, dtype=lr.dtype)
        if noise_vec is None:
            noise_vec = torch.randn(b, self.noise_embed_dim, device=lr.device, dtype=lr.dtype)

        cond = self.cond_mlp(noise_vec)
        x = self.in_conv(torch.cat([x_up, noise_img], dim=1))

        skips = []
        for block in self.downs:
            x = block(x, cond)
            skips.append(x)

        x = self.mid1(x, cond)
        x = self.mid2(x, cond)

        for block in self.ups:
            skip = skips.pop()
            if block.up:
                x = F.interpolate(x, scale_factor=2.0, mode="bilinear", align_corners=False)
            if skip.shape[-2:] != x.shape[-2:]:
                skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)

            x = torch.cat([x, skip], dim=1)
            old_up = block.up
            block.up = False
            x = block(x, cond)
            block.up = old_up

        residual = self.out_conv(self.out_act(self.out_norm(x)))
        return x_up + self.residual_out_scale * residual, x_up


# ============================================================
# Frozen VGG feature extractor
# ============================================================
class FrozenVGGFeatureMaps(nn.Module):
    def __init__(self):
        super().__init__()
        weights = models.VGG16_Weights.IMAGENET1K_V1
        vgg = models.vgg16(weights=weights).features

        self.blocks = nn.ModuleList([
            vgg[:4],
            vgg[4:9],
            vgg[9:16],
            vgg[16:23],
        ])

        for p in self.blocks.parameters():
            p.requires_grad = False
        self.eval()

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        outs = []
        h = x
        for block in self.blocks:
            h = block(h)
            outs.append(h)
        return outs


# ============================================================
# EMA
# ============================================================
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.shadow.state_dict().items():
            if not torch.is_floating_point(v):
                v.copy_(msd[k])
            else:
                v.copy_(v * self.decay + msd[k].detach() * (1.0 - self.decay))


# ============================================================
# Drift helpers
# ============================================================
def get_lambda_drift(epoch: int, cfg: Config) -> float:
    if epoch <= cfg.warmup_epochs:
        return 0.0
    ramp_start = cfg.warmup_epochs + 1
    ramp_end = cfg.warmup_epochs + cfg.drift_ramp_epochs
    if epoch <= ramp_end:
        progress = (epoch - ramp_start + 1) / max(1, cfg.drift_ramp_epochs)
        return cfg.drift_loss_weight + progress * (cfg.drift_loss_weight_max - cfg.drift_loss_weight)
    return cfg.drift_loss_weight_max


def get_lambda_same_neg(epoch: int, cfg: Config) -> float:
    if epoch <= cfg.warmup_epochs:
        return cfg.lambda_same_neg_start
    ramp_end = cfg.warmup_epochs + cfg.drift_ramp_epochs
    if epoch <= ramp_end:
        progress = (epoch - cfg.warmup_epochs) / max(1, cfg.drift_ramp_epochs)
        return cfg.lambda_same_neg_start + progress * (cfg.lambda_same_neg_end - cfg.lambda_same_neg_start)
    return cfg.lambda_same_neg_end


def subsample_spatial_positions(x: torch.Tensor, max_positions: int):
    """
    x: [T, C, H, W] -> [T, C, L_sub], idx
    """
    t, c, h, w = x.shape
    l = h * w
    x = x.view(t, c, l)
    if l <= max_positions:
        return x, None
    idx = torch.randperm(l, device=x.device)[:max_positions]
    return x[:, :, idx], idx


def compute_feature_scale_from_banks(g_feats: torch.Tensor, p_feats: torch.Tensor, eps: float = 1e-6):
    """
    Scalar normalization:
    scale = mean pairwise distance / sqrt(C)

    g_feats: [N, C, L]
    p_feats: [M, C, L]
    """
    c = g_feats.shape[1]
    g_bank = g_feats.permute(0, 2, 1).reshape(-1, c)   # [N*L, C]
    p_bank = p_feats.permute(0, 2, 1).reshape(-1, c)   # [M*L, C]

    if g_bank.numel() == 0 or p_bank.numel() == 0:
        return g_feats.new_tensor(1.0)

    d = torch.cdist(g_bank, p_bank, p=2)
    scale = d.mean() / math.sqrt(c)
    return scale.detach().clamp_min(eps)


def compute_simple_conditional_drift(
    x: torch.Tensor,
    y_pos: torch.Tensor,
    tau: float,
    lambda_pos: float,
    lambda_same_neg: float,
):
    """
    x: [N, C] generated samples at one location
    y_pos: [M, C] positive bank at the same location
    negatives: other generated samples from same LR condition
    """
    dist_pos = torch.cdist(x, y_pos, p=2)
    logit_pos = -dist_pos / tau
    w_pos = torch.softmax(logit_pos, dim=1)
    mu_pos = w_pos @ y_pos
    v_pos = mu_pos - x

    if x.shape[0] > 1:
        dist_same = torch.cdist(x, x, p=2)
        dist_same = dist_same + torch.eye(x.shape[0], device=x.device, dtype=x.dtype) * 1e6
        logit_same = -dist_same / tau
        w_same = torch.softmax(logit_same, dim=1)
        mu_same = w_same @ x
        v_same = mu_same - x
        a_same_mean = float(w_same.mean().detach().cpu())
    else:
        v_same = torch.zeros_like(x)
        a_same_mean = 0.0

    v = lambda_pos * v_pos - lambda_same_neg * v_same

    info = {
        "A_pos_mean": float(w_pos.mean().detach().cpu()),
        "A_same_mean": a_same_mean,
        "V_rms_before_norm": float(torch.sqrt(torch.mean(v ** 2) + 1e-8).detach().cpu()),
    }
    return v, info


class SingleLevelConditionalDriftingLoss(nn.Module):
    """
    One VGG level only.

    feat_scale:
      batch-level scalar normalization across all LR-conditions in the batch

    drift_scale:
      per-LR-condition scalar estimate, but used softly:
      we DO NOT normalize everything to unit RMS.
      Small drift is preserved; only too-large drift is mildly suppressed.
    """
    def __init__(self, encoder: FrozenVGGFeatureMaps, cfg: Config):
        super().__init__()
        self.encoder = encoder
        self.cfg = cfg

    def forward(self, x_gen: torch.Tensor, x_pos: torch.Tensor, x_up: torch.Tensor, epoch: int):
        """
        x_gen: [B, N, C, H, W]
        x_pos: [B, M, C, H, W]
        x_up:  [B, C, H, W]
        """
        B, N, C, H, W = x_gen.shape
        M = x_pos.shape[1]
        lambda_same_neg = get_lambda_same_neg(epoch, self.cfg)

        if self.cfg.drift_fp32:
            x_gen = x_gen.float()
            x_pos = x_pos.float()
            x_up = x_up.float()

        x_gen_flat = x_gen.reshape(B * N, C, H, W)
        x_pos_flat = x_pos.reshape(B * M, C, H, W)
        x_up_flat_for_gen = x_up[:, None].expand(-1, N, -1, -1, -1).reshape(B * N, C, H, W)
        x_up_flat_for_pos = x_up[:, None].expand(-1, M, -1, -1, -1).reshape(B * M, C, H, W)

        gen_maps = self.encoder(x_gen_flat)
        pos_maps = self.encoder(x_pos_flat)
        up_maps_gen = self.encoder(x_up_flat_for_gen)
        up_maps_pos = self.encoder(x_up_flat_for_pos)

        s = self.cfg.feature_scale_index
        g_map = gen_maps[s] - up_maps_gen[s]
        p_map = pos_maps[s] - up_maps_pos[s]

        _, Cg, Hs, Ws = g_map.shape
        g_map = g_map.view(B, N, Cg, Hs, Ws)
        p_map = p_map.view(B, M, Cg, Hs, Ws)

        # --------------------------------------------------
        # 1) collect subsampled features for each LR-condition
        # --------------------------------------------------
        g_list, p_list = [], []
        for b in range(B):
            g_b, idx = subsample_spatial_positions(g_map[b], self.cfg.feature_max_positions)  # [N,C,L]
            p_full = p_map[b].view(M, Cg, Hs * Ws)
            p_b = p_full if idx is None else p_full[:, :, idx]  # [M,C,L]
            g_list.append(g_b)
            p_list.append(p_b)

        # --------------------------------------------------
        # 2) ONE feat_scale for the whole batch
        # --------------------------------------------------
        g_batch = torch.cat(g_list, dim=2)  # [N,C,sumL]
        p_batch = torch.cat(p_list, dim=2)  # [M,C,sumL]

        feat_scale = compute_feature_scale_from_banks(
            g_batch, p_batch, eps=self.cfg.norm_eps
        ).clamp_min(self.cfg.feat_scale_min)

        g_list = [g_b / feat_scale for g_b in g_list]
        p_list = [p_b / feat_scale for p_b in p_list]

        total_loss = x_gen.new_tensor(0.0)
        num_items = 0

        logs_a_pos, logs_a_same, logs_v_rms = [], [], []
        logs_feat_scale = [float(feat_scale.detach().cpu())]
        logs_drift_scale = []
        logs_drift_norm_factor = []

        # --------------------------------------------------
        # 3) soft drift normalization per LR-condition
        # --------------------------------------------------
        for b in range(B):
            g_b = g_list[b]
            p_b = p_list[b]
            L = g_b.shape[-1]

            raw_vs = []
            infos = []

            for l in range(L):
                x_loc = g_b[:, :, l]
                p_loc = p_b[:, :, l]

                v_loc, info = compute_simple_conditional_drift(
                    x=x_loc,
                    y_pos=p_loc,
                    tau=self.cfg.temperature,
                    lambda_pos=self.cfg.lambda_pos,
                    lambda_same_neg=lambda_same_neg,
                )
                raw_vs.append(v_loc)
                infos.append(info)

            v_stack = torch.stack(raw_vs, dim=0)  # [L,N,C]
            drift_scale = torch.sqrt(torch.mean(v_stack ** 2) + self.cfg.norm_eps).detach()

            # soft normalization:
            # do not amplify small drift; only suppress too-large drift
            drift_norm_factor = drift_scale.clamp(
                min=self.cfg.drift_scale_center,
                max=self.cfg.drift_scale_max,
            )

            logs_drift_scale.append(float(drift_scale.cpu()))
            logs_drift_norm_factor.append(float(drift_norm_factor.cpu()))

            for l in range(L):
                x_loc = g_b[:, :, l]
                v_loc = raw_vs[l] / drift_norm_factor

                target = (x_loc + self.cfg.drift_step_size * v_loc).detach()
                loc_loss = F.mse_loss(x_loc, target)

                total_loss = total_loss + loc_loss
                num_items += 1

                logs_a_pos.append(infos[l]["A_pos_mean"])
                logs_a_same.append(infos[l]["A_same_mean"])
                logs_v_rms.append(infos[l]["V_rms_before_norm"])

        total_loss = total_loss / max(1, num_items)

        info = {
            "lambda_same_neg": float(lambda_same_neg),
            "scale": s,
            "A_pos_mean": float(np.mean(logs_a_pos)) if logs_a_pos else float("nan"),
            "A_same_mean": float(np.mean(logs_a_same)) if logs_a_same else float("nan"),
            "V_rms": float(np.mean(logs_v_rms)) if logs_v_rms else float("nan"),
            "feat_scale": float(np.mean(logs_feat_scale)) if logs_feat_scale else float("nan"),
            "drift_scale": float(np.mean(logs_drift_scale)) if logs_drift_scale else float("nan"),
            "drift_norm_factor": float(np.mean(logs_drift_norm_factor)) if logs_drift_norm_factor else float("nan"),
        }
        return total_loss, info


# ============================================================
# Train helpers
# ============================================================
def generate_multi_samples(model: nn.Module, lr: torch.Tensor, cfg: Config, num_samples: Optional[int] = None):
    if num_samples is None:
        num_samples = cfg.num_samples_per_lr

    B, C, h_lr, w_lr = lr.shape
    H, W = h_lr * cfg.scale, w_lr * cfg.scale

    lr_rep = lr[:, None].repeat(1, num_samples, 1, 1, 1).reshape(B * num_samples, C, h_lr, w_lr)
    noise_img = torch.randn(B * num_samples, cfg.noise_image_channels, H, W, device=lr.device, dtype=lr.dtype)
    noise_vec = torch.randn(B * num_samples, cfg.noise_embed_dim, device=lr.device, dtype=lr.dtype)

    x_hat_rep, x_up_rep = model(lr_rep, noise_img=noise_img, noise_vec=noise_vec)
    x_gen = x_hat_rep.view(B, num_samples, cfg.in_channels, H, W)
    x_up = x_up_rep.view(B, num_samples, cfg.in_channels, H, W)[:, 0]
    return x_gen, x_up


@torch.no_grad()
def sample_sr(model: nn.Module, lr: torch.Tensor, cfg: Config, zero_noise: bool = True, return_up: bool = False):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    if zero_noise:
        noise_img = torch.zeros(B, cfg.noise_image_channels, H, W, device=lr.device, dtype=lr.dtype)
        noise_vec = torch.zeros(B, cfg.noise_embed_dim, device=lr.device, dtype=lr.dtype)
    else:
        noise_img = torch.randn(B, cfg.noise_image_channels, H, W, device=lr.device, dtype=lr.dtype)
        noise_vec = torch.randn(B, cfg.noise_embed_dim, device=lr.device, dtype=lr.dtype)

    x_hat, x_up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return (x_hat, x_up) if return_up else x_hat


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, cfg: Config, lpips_model=None, max_batches: int = 20):
    model.eval()

    psnr_model, psnr_bic = [], []
    lpips_model_vals, lpips_bic_vals = [], []

    amp_device_type = "cuda" if "cuda" in cfg.device else "cpu"

    for i, (hr, lr) in enumerate(loader):
        if i >= max_batches:
            break

        hr = hr.to(cfg.device, non_blocking=True)
        lr = lr.to(cfg.device, non_blocking=True)

        with autocast(device_type=amp_device_type, enabled=(cfg.use_amp and "cuda" in cfg.device)):
            pred, up = sample_sr(model, lr, cfg, zero_noise=cfg.eval_use_zero_noise, return_up=True)

        pred = pred.clamp(0.0, 1.0)
        up = up.clamp(0.0, 1.0)

        psnr_model.append(calc_psnr_sr(pred, hr, shave=cfg.shave_border, use_y=cfg.eval_on_y_channel))
        psnr_bic.append(calc_psnr_sr(up, hr, shave=cfg.shave_border, use_y=cfg.eval_on_y_channel))

        if lpips_model is not None:
            lpips_model_vals.append(calc_lpips_sr(pred, hr, lpips_model, shave=cfg.eval_lpips_shave_border))
            lpips_bic_vals.append(calc_lpips_sr(up, hr, lpips_model, shave=cfg.eval_lpips_shave_border))

        del pred, up, hr, lr

    model.train()
    return {
        "psnr_model": float(np.mean(psnr_model)) if psnr_model else float("nan"),
        "psnr_bicubic": float(np.mean(psnr_bic)) if psnr_bic else float("nan"),
        "lpips_model": float(np.mean(lpips_model_vals)) if lpips_model_vals else float("nan"),
        "lpips_bicubic": float(np.mean(lpips_bic_vals)) if lpips_bic_vals else float("nan"),
    }


# ============================================================
# Checkpoint/history
# ============================================================
def get_rng_state():
    out = {
        "python_random": random.getstate(),
        "numpy_random": np.random.get_state(),
        "torch_random": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        out["torch_cuda_random"] = torch.cuda.get_rng_state_all()
    return out


def atomic_torch_save(obj, path: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)


def make_experiment_name(cfg: Config) -> str:
    return (
        f"drift_sr_batchfeat_softdrift"
        f"_s{cfg.feature_scale_index}"
        f"_tau{cfg.temperature}"
        f"_ns{cfg.num_samples_per_lr}"
        f"_np{cfg.num_positive_views}"
        f"_seed{cfg.seed}"
    )


def save_checkpoint(
    model,
    ema,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    best_psnr,
    history,
    cfg: Config,
    tag: str,
):
    path = os.path.join(cfg.ckpt_dir, f"{tag}.pt")
    atomic_torch_save({
        "epoch": epoch,
        "global_step": global_step,
        "best_psnr": best_psnr,
        "model_state": model.state_dict(),
        "ema_state": ema.shadow.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "config": asdict(cfg),
        "rng_state": get_rng_state(),
    }, path)
    print(f"[ckpt] saved: {path}")


def save_history_json(history: Dict, cfg: Config, final_stats: Optional[Dict] = None):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(cfg.history_dir, f"{make_experiment_name(cfg)}_{timestamp}.json")
    payload = {
        "experiment_name": make_experiment_name(cfg),
        "timestamp": timestamp,
        "config": asdict(cfg),
        "history": history,
        "final_stats": final_stats,
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"[history] saved: {path}")
    return path


# ============================================================
# Plotting
# ============================================================
def _epoch_mean_from_steps(values: List[float], steps_per_epoch: int, n_eval_points: int) -> List[float]:
    if steps_per_epoch <= 0:
        return []
    out = []
    for start in range(0, len(values), steps_per_epoch):
        chunk = values[start:start + steps_per_epoch]
        if len(chunk) > 0:
            out.append(float(np.mean(chunk)))
    return out[:n_eval_points]


def plot_training_curves(history: Dict, cfg: Config, steps_per_epoch: int):
    eval_epochs = history["eval_epochs"]
    n_eval = len(eval_epochs)

    train_total_epoch = _epoch_mean_from_steps(history["train_total"], steps_per_epoch, n_eval)
    train_pix_epoch = _epoch_mean_from_steps(history["train_pix"], steps_per_epoch, n_eval)
    train_lr_epoch = _epoch_mean_from_steps(history["train_lr_cons"], steps_per_epoch, n_eval)
    train_drift_epoch = _epoch_mean_from_steps(history["train_drift"], steps_per_epoch, n_eval)

    saved_paths = []

    plt.figure(figsize=(8, 5))
    plt.plot(eval_epochs, history["val_psnr"], marker="o", label="Val PSNR")
    plt.plot(eval_epochs, history["val_psnr_bicubic"], marker="o", label="Bicubic PSNR")
    plt.xlabel("Epoch")
    plt.ylabel("PSNR")
    plt.title("Validation PSNR vs Epoch")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(cfg.plot_dir, "val_psnr_vs_epoch.png")
    plt.savefig(path, dpi=180)
    plt.close()
    saved_paths.append(path)

    plt.figure(figsize=(8, 5))
    plt.plot(eval_epochs, history["val_lpips"], marker="o", label="Val LPIPS")
    plt.plot(eval_epochs, history["val_lpips_bicubic"], marker="o", label="Bicubic LPIPS")
    plt.xlabel("Epoch")
    plt.ylabel("LPIPS")
    plt.title("Validation LPIPS vs Epoch")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(cfg.plot_dir, "val_lpips_vs_epoch.png")
    plt.savefig(path, dpi=180)
    plt.close()
    saved_paths.append(path)

    plt.figure(figsize=(8, 5))
    plt.plot(eval_epochs, train_total_epoch, marker="o", label="Train total")
    plt.plot(eval_epochs, train_pix_epoch, marker="o", label="Train pixel")
    plt.plot(eval_epochs, train_lr_epoch, marker="o", label="Train LR consistency")
    plt.plot(eval_epochs, train_drift_epoch, marker="o", label="Train drift raw")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Losses vs Epoch")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(cfg.plot_dir, "train_losses_vs_epoch.png")
    plt.savefig(path, dpi=180)
    plt.close()
    saved_paths.append(path)

    return saved_paths


# ============================================================
# Train
# ============================================================
def train(cfg: Config):
    set_seed(cfg.seed)
    ensure_dirs(cfg)

    print("Device:", cfg.device)
    print("CUDA available:", torch.cuda.is_available())
    print("Experiment:", make_experiment_name(cfg))

    train_ds = DIV2KPairDataset(
        cfg.train_hr_dir, cfg.train_lr_dir,
        scale=cfg.scale, patch_size=cfg.patch_size, training=True
    )
    val_ds = DIV2KPairDataset(
        cfg.val_hr_dir, cfg.val_lr_dir,
        scale=cfg.scale, patch_size=None, training=False
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
        prefetch_factor=2 if cfg.num_workers > 0 else None,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.val_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
        prefetch_factor=2 if cfg.num_workers > 0 else None,
    )

    model = NoiseConditionalResidualUNetSR(
        in_channels=cfg.in_channels,
        noise_image_channels=cfg.noise_image_channels,
        base=cfg.unet_base,
        channel_mult=cfg.unet_channel_mult,
        num_blocks=cfg.unet_num_blocks,
        noise_embed_dim=cfg.noise_embed_dim,
        residual_out_scale=cfg.residual_out_scale,
        scale=cfg.scale,
    ).to(cfg.device)

    frozen_encoder = FrozenVGGFeatureMaps().to(cfg.device)
    for p in frozen_encoder.parameters():
        p.requires_grad = False
    frozen_encoder.eval()

    drift_loss_fn = SingleLevelConditionalDriftingLoss(frozen_encoder, cfg)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.min_lr)

    amp_device_type = "cuda" if "cuda" in cfg.device else "cpu"
    scaler = GradScaler(device=amp_device_type, enabled=(cfg.use_amp and "cuda" in cfg.device))

    ema = EMA(model, decay=cfg.ema_decay)

    lpips_model = lpips.LPIPS(net=cfg.eval_lpips_net).to(cfg.device).eval()
    for p in lpips_model.parameters():
        p.requires_grad = False

    history = {
        "train_total": [],
        "train_pix": [],
        "train_lr_cons": [],
        "train_drift": [],
        "lambda_drift": [],
        "lambda_same_neg": [],
        "eval_epochs": [],
        "val_psnr": [],
        "val_psnr_bicubic": [],
        "val_lpips": [],
        "val_lpips_bicubic": [],
    }

    global_step = 0
    best_psnr = -1e9

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        lambda_drift = get_lambda_drift(epoch, cfg)
        lambda_same_neg = get_lambda_same_neg(epoch, cfg)

        running_total, running_pix, running_lr, running_drift = [], [], [], []

        for hr, lr in train_loader:
            hr = hr.to(cfg.device, non_blocking=True)
            lr = lr.to(cfg.device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=amp_device_type, enabled=(cfg.use_amp and "cuda" in cfg.device)):
                x_gen, x_up = generate_multi_samples(model, lr, cfg, num_samples=cfg.num_samples_per_lr)
                positive_bank = build_positive_bank(hr, lr, cfg)

                # random anchor instead of always first sample
                anchor_idx = random.randrange(x_gen.shape[1])
                x_anchor = x_gen[:, anchor_idx]
                loss_pix = cfg.pixel_l1_w * F.l1_loss(x_anchor, hr)

                B, N, C, H, W = x_gen.shape
                x_gen_lr = degrade_to_lr(x_gen.view(B * N, C, H, W), scale=cfg.scale)
                loss_lr = cfg.lr_consistency_w * F.l1_loss(
                    x_gen_lr.view(B, N, C, H // cfg.scale, W // cfg.scale),
                    lr[:, None].expand(-1, N, -1, -1, -1),
                )

                if lambda_drift > 0.0:
                    drift_raw, drift_info = drift_loss_fn(
                        x_gen=x_gen,
                        x_pos=positive_bank,
                        x_up=x_up,
                        epoch=epoch,
                    )
                    loss_drift = lambda_drift * drift_raw
                else:
                    drift_raw = hr.new_tensor(0.0)
                    drift_info = {
                        "lambda_same_neg": float(lambda_same_neg),
                        "scale": cfg.feature_scale_index,
                        "A_pos_mean": float("nan"),
                        "A_same_mean": float("nan"),
                        "V_rms": float("nan"),
                        "feat_scale": float("nan"),
                        "drift_scale": float("nan"),
                        "drift_norm_factor": float("nan"),
                    }
                    loss_drift = hr.new_tensor(0.0)

                total_loss = loss_pix + loss_lr + loss_drift

            if not torch.isfinite(total_loss):
                print(f"[warn] non-finite total loss at epoch {epoch}, step {global_step}; batch skipped")
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            if not torch.isfinite(grad_norm):
                print(f"[warn] non-finite grad norm at epoch {epoch}, step {global_step}; optimizer step skipped")
                optimizer.zero_grad(set_to_none=True)
                scaler.update()
                continue

            scaler.step(optimizer)
            scaler.update()
            ema.update(model)

            history["train_total"].append(float(total_loss.item()))
            history["train_pix"].append(float(loss_pix.item()))
            history["train_lr_cons"].append(float(loss_lr.item()))
            history["train_drift"].append(float(drift_raw.item()))
            history["lambda_drift"].append(float(lambda_drift))
            history["lambda_same_neg"].append(float(lambda_same_neg))

            running_total.append(float(total_loss.item()))
            running_pix.append(float(loss_pix.item()))
            running_lr.append(float(loss_lr.item()))
            running_drift.append(float(drift_raw.item()))

            if global_step % cfg.log_every == 0:
                print(
                    f"[train] ep {epoch} step {global_step}"
                    f" | total={total_loss.item():.5f}"
                    f" | pix={loss_pix.item():.5f}"
                    f" | lr={loss_lr.item():.5f}"
                    f" | drift_raw={drift_raw.item():.5f}"
                    f" | lambda_drift={lambda_drift:.3f}"
                    f" | lambda_same_neg={lambda_same_neg:.3f}"
                    f" | grad={float(grad_norm):.4f}"
                    f" | lr_now={optimizer.param_groups[0]['lr']:.3e}"
                    f" | s{drift_info['scale']}_Apos={drift_info['A_pos_mean']:.4f}"
                    f" | s{drift_info['scale']}_Asame={drift_info['A_same_mean']:.4f}"
                    f" | s{drift_info['scale']}_Vrms={drift_info['V_rms']:.4f}"
                    f" | s{drift_info['scale']}_Fscale={drift_info['feat_scale']:.4f}"
                    f" | s{drift_info['scale']}_Dscale={drift_info['drift_scale']:.4f}"
                    f" | s{drift_info['scale']}_Dnorm={drift_info['drift_norm_factor']:.4f}"
                )
            global_step += 1

        scheduler.step()

        if (epoch % cfg.eval_every_epochs == 0) or (epoch == cfg.epochs):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            stats = evaluate(
                ema.shadow,
                val_loader,
                cfg,
                lpips_model=lpips_model,
                max_batches=cfg.val_batches,
            )

            history["eval_epochs"].append(epoch)
            history["val_psnr"].append(stats["psnr_model"])
            history["val_psnr_bicubic"].append(stats["psnr_bicubic"])
            history["val_lpips"].append(stats["lpips_model"])
            history["val_lpips_bicubic"].append(stats["lpips_bicubic"])

            print(
                f"[val {epoch}]"
                f" psnr={stats['psnr_model']:.3f}"
                f" | bicubic_psnr={stats['psnr_bicubic']:.3f}"
                f" | lpips={stats['lpips_model']:.4f}"
                f" | bicubic_lpips={stats['lpips_bicubic']:.4f}"
                f" | mean_total={np.mean(running_total) if running_total else float('nan'):.5f}"
                f" | mean_pix={np.mean(running_pix) if running_pix else float('nan'):.5f}"
                f" | mean_lr={np.mean(running_lr) if running_lr else float('nan'):.5f}"
                f" | mean_drift={np.mean(running_drift) if running_drift else float('nan'):.5f}"
            )

            if stats["psnr_model"] > best_psnr:
                best_psnr = stats["psnr_model"]
                save_checkpoint(
                    model, ema, optimizer, scheduler, scaler,
                    epoch, global_step, best_psnr, history, cfg, "best_psnr"
                )

            save_checkpoint(
                model, ema, optimizer, scheduler, scaler,
                epoch, global_step, best_psnr, history, cfg, "last"
            )

            if epoch % cfg.save_every_epochs == 0:
                save_checkpoint(
                    model, ema, optimizer, scheduler, scaler,
                    epoch, global_step, best_psnr, history, cfg, f"epoch_{epoch:03d}"
                )

    final_stats = evaluate(
        ema.shadow,
        val_loader,
        cfg,
        lpips_model=lpips_model,
        max_batches=cfg.val_batches,
    )

    history_path = save_history_json(history, cfg, final_stats=final_stats)
    plot_paths = plot_training_curves(history, cfg, steps_per_epoch=len(train_loader))

    print("Final stats:", final_stats)
    print("Best PSNR:", best_psnr)
    print("History saved to:", history_path)
    print("Plots:")
    for p in plot_paths:
        print(" -", p)

    return ema.shadow, history, final_stats, plot_paths



In [3]:
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import lpips

# =========================
# настройки
# =========================
CHECKPOINT_PATH = "/kaggle/input/datasets/ksenia1589/epoch-30/epoch_030.pt"
OUTPUT_DIR = "/kaggle/working/sr_30"

USE_VAL_DATASET = True
SAVE_CSV = True
SAVE_JSON = True

# deterministic метрики к GT
USE_ZERO_NOISE_FOR_FIDELITY = True

# stochastic seeds только для pair-метрик
PAIR_NOISE_SEEDS = [0, 1, 2]   # 3 seeds -> 3 pair'а на изображение
PROGRESS_EVERY = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = cfg.device if torch.cuda.is_available() else "cpu"

# =========================
# dataset
# =========================
dataset = DIV2KPairDataset(
    hr_dir=cfg.val_hr_dir if USE_VAL_DATASET else cfg.train_hr_dir,
    lr_dir=cfg.val_lr_dir if USE_VAL_DATASET else cfg.train_lr_dir,
    scale=cfg.scale,
    patch_size=None,
    training=False,
)

print(f"Dataset size: {len(dataset)}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Pair noise seeds: {PAIR_NOISE_SEEDS} -> {len(PAIR_NOISE_SEEDS) * (len(PAIR_NOISE_SEEDS)-1) // 2} pairs/image")

# =========================
# model + checkpoint
# =========================
model = NoiseConditionalResidualUNetSR(
    in_channels=cfg.in_channels,
    noise_image_channels=cfg.noise_image_channels,
    base=cfg.unet_base,
    channel_mult=cfg.unet_channel_mult,
    num_blocks=cfg.unet_num_blocks,
    noise_embed_dim=cfg.noise_embed_dim,
    residual_out_scale=cfg.residual_out_scale,
    scale=cfg.scale,
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
state = ckpt["ema_state"] if "ema_state" in ckpt else ckpt["model_state"]
model.load_state_dict(state, strict=True)
model.eval()

# =========================
# LPIPS
# =========================
lpips_fn = lpips.LPIPS(net="vgg").to(device).eval()
for p in lpips_fn.parameters():
    p.requires_grad = False

# =========================
# helpers
# =========================
@torch.no_grad()
def calc_lpips01(x, y):
    x_ = x.clamp(0, 1) * 2 - 1
    y_ = y.clamp(0, 1) * 2 - 1
    return float(lpips_fn(x_, y_).mean().item())

@torch.no_grad()
def sample_sr_no_noise(model, lr):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    noise_img = torch.zeros(
        B, cfg.noise_image_channels, H, W,
        device=lr.device, dtype=lr.dtype
    )
    noise_vec = torch.zeros(
        B, cfg.noise_embed_dim,
        device=lr.device, dtype=lr.dtype
    )

    pred, up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return pred.clamp(0, 1), up.clamp(0, 1)

@torch.no_grad()
def sample_sr_with_noise_seed(model, lr, cfg, noise_seed):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    gen = torch.Generator(device=lr.device)
    gen.manual_seed(int(noise_seed))

    noise_img = torch.randn(
        B, cfg.noise_image_channels, H, W,
        device=lr.device, dtype=lr.dtype, generator=gen
    )
    noise_vec = torch.randn(
        B, cfg.noise_embed_dim,
        device=lr.device, dtype=lr.dtype, generator=gen
    )

    pred, up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return pred.clamp(0, 1), up.clamp(0, 1)

def print_running_stats(df_fidelity, df_pairs, processed, total):
    msg = [f"[{processed}/{total}]"]

    if len(df_fidelity) > 0:
        msg.append(
            "fidelity:"
            f" psnr={df_fidelity['psnr_pred'].mean():.4f},"
            f" lpips={df_fidelity['lpips_pred'].mean():.6f},"
            f" bic_psnr={df_fidelity['psnr_bicubic'].mean():.4f},"
            f" bic_lpips={df_fidelity['lpips_bicubic'].mean():.6f}"
        )

    if len(df_pairs) > 0:
        msg.append(
            "pairs:"
            f" pair_lpips={df_pairs['pair_lpips'].mean():.6f},"
            f" pair_l1={df_pairs['pair_l1'].mean():.6f},"
            f" pair_residual_l1={df_pairs['pair_residual_l1'].mean():.6f},"
            f" pair_down_l1={df_pairs['pair_down_l1'].mean():.6f}"
        )

    print(" | ".join(msg))

# =========================
# eval
# =========================
rows_fidelity = []
rows_pairs = []

for ds_idx in range(len(dataset)):
    hr, lr = dataset[ds_idx]
    hr = hr.to(device)
    lr = lr.to(device)

    hr_in = hr.unsqueeze(0)
    lr_in = lr.unsqueeze(0)

    # -------------------------
    # 1) deterministic fidelity
    # -------------------------
    if USE_ZERO_NOISE_FOR_FIDELITY:
        pred_det, bic = sample_sr_no_noise(model, lr_in)
    else:
        pred_det, bic = sample_sr_with_noise_seed(model, lr_in, cfg, noise_seed=PAIR_NOISE_SEEDS[0])

    psnr_pred = calc_psnr_sr(
        pred_det, hr_in,
        shave=cfg.shave_border,
        use_y=cfg.eval_on_y_channel
    )
    psnr_bic = calc_psnr_sr(
        bic, hr_in,
        shave=cfg.shave_border,
        use_y=cfg.eval_on_y_channel
    )

    lpips_pred = calc_lpips01(pred_det, hr_in)
    lpips_bic = calc_lpips01(bic, hr_in)

    rows_fidelity.append({
        "checkpoint": Path(CHECKPOINT_PATH).name,
        "dataset_index": ds_idx,
        "psnr_pred": psnr_pred,
        "psnr_bicubic": psnr_bic,
        "lpips_pred": lpips_pred,
        "lpips_bicubic": lpips_bic,
    })

    # -------------------------
    # 2) stochastic pair metrics
    # -------------------------
    preds = []
    for noise_seed in PAIR_NOISE_SEEDS:
        pred_s, _ = sample_sr_with_noise_seed(model, lr_in, cfg, noise_seed=noise_seed)
        preds.append((int(noise_seed), pred_s))

    for (seed_a, pred_a), (seed_b, pred_b) in itertools.combinations(preds, 2):
        pair_lpips = calc_lpips01(pred_a, pred_b)
        pair_l1 = float(F.l1_loss(pred_a, pred_b).item())

        res_a = pred_a - bic
        res_b = pred_b - bic
        pair_residual_l1 = float(F.l1_loss(res_a, res_b).item())

        down_a = F.interpolate(pred_a, size=lr_in.shape[-2:], mode="bicubic", align_corners=False)
        down_b = F.interpolate(pred_b, size=lr_in.shape[-2:], mode="bicubic", align_corners=False)
        pair_down_l1 = float(F.l1_loss(down_a, down_b).item())

        rows_pairs.append({
            "checkpoint": Path(CHECKPOINT_PATH).name,
            "dataset_index": ds_idx,
            "seed_a": int(seed_a),
            "seed_b": int(seed_b),
            "pair_lpips": pair_lpips,
            "pair_l1": pair_l1,
            "pair_residual_l1": pair_residual_l1,
            "pair_down_l1": pair_down_l1,
        })

    # -------------------------
    # progress prints
    # -------------------------
    processed = ds_idx + 1
    if processed % PROGRESS_EVERY == 0 or processed == len(dataset):
        df_fid_tmp = pd.DataFrame(rows_fidelity)
        df_pair_tmp = pd.DataFrame(rows_pairs)
        print_running_stats(df_fid_tmp, df_pair_tmp, processed, len(dataset))

# =========================
# DataFrames
# =========================
df_fidelity = pd.DataFrame(rows_fidelity)
df_pairs = pd.DataFrame(rows_pairs)

# =========================
# summary per image
# =========================
summary_pairs_per_image = (
    df_pairs
    .groupby(["dataset_index"])
    .agg(
        pair_lpips_mean=("pair_lpips", "mean"),
        pair_lpips_std=("pair_lpips", "std"),
        pair_l1_mean=("pair_l1", "mean"),
        pair_l1_std=("pair_l1", "std"),
        pair_residual_l1_mean=("pair_residual_l1", "mean"),
        pair_residual_l1_std=("pair_residual_l1", "std"),
        pair_down_l1_mean=("pair_down_l1", "mean"),
        pair_down_l1_std=("pair_down_l1", "std"),
    )
    .reset_index()
)

summary_per_image = df_fidelity.merge(summary_pairs_per_image, on="dataset_index", how="left")

# =========================
# global summary
# =========================
summary = pd.DataFrame([{
    "checkpoint": Path(CHECKPOINT_PATH).name,
    "num_images": int(len(df_fidelity)),
    "pair_seeds": str(PAIR_NOISE_SEEDS),
    "num_pairs_per_image": int(len(PAIR_NOISE_SEEDS) * (len(PAIR_NOISE_SEEDS)-1) // 2),

    "mean_psnr_pred": float(df_fidelity["psnr_pred"].mean()),
    "mean_psnr_bicubic": float(df_fidelity["psnr_bicubic"].mean()),
    "mean_lpips_pred": float(df_fidelity["lpips_pred"].mean()),
    "mean_lpips_bicubic": float(df_fidelity["lpips_bicubic"].mean()),

    "mean_pair_lpips": float(df_pairs["pair_lpips"].mean()),
    "std_pair_lpips": float(df_pairs["pair_lpips"].std()),
    "mean_pair_l1": float(df_pairs["pair_l1"].mean()),
    "std_pair_l1": float(df_pairs["pair_l1"].std()),
    "mean_pair_residual_l1": float(df_pairs["pair_residual_l1"].mean()),
    "std_pair_residual_l1": float(df_pairs["pair_residual_l1"].std()),
    "mean_pair_down_l1": float(df_pairs["pair_down_l1"].mean()),
    "std_pair_down_l1": float(df_pairs["pair_down_l1"].std()),
}])

# =========================
# save
# =========================
if SAVE_CSV:
    fidelity_csv = os.path.join(OUTPUT_DIR, "full_val_fidelity_no_noise.csv")
    pairs_csv = os.path.join(OUTPUT_DIR, "full_val_pair_metrics.csv")
    per_image_csv = os.path.join(OUTPUT_DIR, "full_val_summary_per_image.csv")
    summary_csv = os.path.join(OUTPUT_DIR, "full_val_summary.csv")

    df_fidelity.to_csv(fidelity_csv, index=False)
    df_pairs.to_csv(pairs_csv, index=False)
    summary_per_image.to_csv(per_image_csv, index=False)
    summary.to_csv(summary_csv, index=False)

if SAVE_JSON:
    summary_json = os.path.join(OUTPUT_DIR, "full_val_summary.json")
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary.iloc[0].to_dict(), f, ensure_ascii=False, indent=2)

# =========================
# print
# =========================
print("\n=== PER-IMAGE FIDELITY ===")
display(df_fidelity)

print("\n=== PER-PAIR METRICS ===")
display(df_pairs)

print("\n=== SUMMARY PER IMAGE ===")
display(summary_per_image)

print("\n=== GLOBAL SUMMARY ===")
display(summary)

print("\nSaved to:", OUTPUT_DIR)


Dataset size: 100
Checkpoint: /kaggle/input/datasets/ksenia1589/epoch-30/epoch_030.pt
Pair noise seeds: [0, 1, 2] -> 3 pairs/image
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 187MB/s] 


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[10/100] | fidelity: psnr=27.8933, lpips=0.319017, bic_psnr=27.7598, bic_lpips=0.345747 | pairs: pair_lpips=0.018470, pair_l1=0.001186, pair_residual_l1=0.001186, pair_down_l1=0.000127
[20/100] | fidelity: psnr=27.9030, lpips=0.319078, bic_psnr=27.7653, bic_lpips=0.343117 | pairs: pair_lpips=0.021358, pair_l1=0.001216, pair_residual_l1=0.001216, pair_down_l1=0.000127
[30/100] | fidelity: psnr=26.9255, lpips=0.327108, bic_psnr=26.7808, bic_lpips=0.351744 | pairs: pair_lpips=0.020075, pair_l1=0.001175, pair_residual_l1=0.001175, pair_down_l1=0.000129
[40/100] | fidelity: psnr=27.0342, lpips=0.330811, bic_psnr=26.8912, bic_lpips=0.352491 | pairs: pair_lpips=0.019996, pair_l1=0.001164, pair_residual_l1=0.001164, pair_down_l1=0.000130
[50/100] | fidelity: psnr=27.1086, lpips=0.327119, bic_psnr=26.9598, bic_lpips=0.348641 | pairs: pair_lpips=0.021141, pair_l1=0.001162, pair_residual_l1=0.001162, pair_down_

...


=== SUMMARY PER IMAGE ===


,checkpoint,dataset_index,psnr_pred,psnr_bicubic,lpips_pred,lpips_bicubic,pair_lpips_mean,pair_lpips_std,pair_l1_mean,pair_l1_std,pair_residual_l1_mean,pair_residual_l1_std,pair_down_l1_mean,pair_down_l1_std
0,epoch_030.pt,0,27.931095,27.784136,0.320715,0.329026,0.009622,0.000419,0.001001,0.000066,0.001001,0.000066,0.000120,0.000006
1,epoch_030.pt,1,33.875275,33.764965,0.188891,0.213919,0.008742,0.000727,0.001321,0.000066,0.001321,0.000066,0.000109,0.000004
2,epoch_030.pt,2,34.392860,33.956917,0.177024,0.228932,0.045273,0.000367,0.002036,0.000032,0.002036,0.000032,0.000153,0.000001
3,epoch_030.pt,3,25.837906,25.704222,0.341303,0.352412,0.009855,0.001451,0.001073,0.000080,0.001073,0.000080,0.000125,0.000008
4,epoch_030.pt,4,27.698509,27.614622,0.390253,0.424505,0.009597,0.000188,0.000944,0.000018,0.000944,0.000018,0.000132,0.000010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,epoch_030.pt,95,34.508537,33.906025,0.343679,0.349816,0.055935,0.004521,0.001278,0.000063,0.001278,0.000063,0.000099,0.000005
96,epoch_030.pt,96,20.854445,20.770344,0.405761,0.404800,0.000975,0.000277,0.000629,0.000052,0.000629,0.000052,0.000155,0.000007
97,epoch_030.pt,97,32.202652,31.961782,0.357263,0.335687,0.037972,0.000927,0.001210,0.000037,0.001210,0.000037,0.000099,0.000002
98,epoch_030.pt,98,26.712820,26.362011,0.250124,0.262098,0.012974,0.001152,0.001118,0.000085,0.001118,0.000085,0.000116,0.000007



=== GLOBAL SUMMARY ===


,checkpoint,num_images,pair_seeds,num_pairs_per_image,mean_psnr_pred,mean_psnr_bicubic,mean_lpips_pred,mean_lpips_bicubic,mean_pair_lpips,std_pair_lpips,mean_pair_l1,std_pair_l1,mean_pair_residual_l1,std_pair_residual_l1,mean_pair_down_l1,std_pair_down_l1
0,epoch_030.pt,100,"[0, 1, 2]",3,27.051296,26.907455,0.340372,0.360611,0.021296,0.015903,0.001151,0.00026,0.001151,0.00026,0.000128,0.00002



Saved to: /kaggle/working/sr_30


In [3]:
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import lpips

# =========================
# настройки
# =========================
CHECKPOINT_PATH = "/kaggle/input/datasets/ksenia1589/epoch-40/epoch_040.pt"
OUTPUT_DIR = "/kaggle/working/sr_40"

USE_VAL_DATASET = True
SAVE_CSV = True
SAVE_JSON = True

# deterministic метрики к GT
USE_ZERO_NOISE_FOR_FIDELITY = True

# stochastic seeds только для pair-метрик
PAIR_NOISE_SEEDS = [0, 1, 2]   # 3 seeds -> 3 pair'а на изображение
PROGRESS_EVERY = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = cfg.device if torch.cuda.is_available() else "cpu"

# =========================
# dataset
# =========================
dataset = DIV2KPairDataset(
    hr_dir=cfg.val_hr_dir if USE_VAL_DATASET else cfg.train_hr_dir,
    lr_dir=cfg.val_lr_dir if USE_VAL_DATASET else cfg.train_lr_dir,
    scale=cfg.scale,
    patch_size=None,
    training=False,
)

print(f"Dataset size: {len(dataset)}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Pair noise seeds: {PAIR_NOISE_SEEDS} -> {len(PAIR_NOISE_SEEDS) * (len(PAIR_NOISE_SEEDS)-1) // 2} pairs/image")

# =========================
# model + checkpoint
# =========================
model = NoiseConditionalResidualUNetSR(
    in_channels=cfg.in_channels,
    noise_image_channels=cfg.noise_image_channels,
    base=cfg.unet_base,
    channel_mult=cfg.unet_channel_mult,
    num_blocks=cfg.unet_num_blocks,
    noise_embed_dim=cfg.noise_embed_dim,
    residual_out_scale=cfg.residual_out_scale,
    scale=cfg.scale,
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
state = ckpt["ema_state"] if "ema_state" in ckpt else ckpt["model_state"]
model.load_state_dict(state, strict=True)
model.eval()

# =========================
# LPIPS
# =========================
lpips_fn = lpips.LPIPS(net="vgg").to(device).eval()
for p in lpips_fn.parameters():
    p.requires_grad = False

# =========================
# helpers
# =========================
@torch.no_grad()
def calc_lpips01(x, y):
    x_ = x.clamp(0, 1) * 2 - 1
    y_ = y.clamp(0, 1) * 2 - 1
    return float(lpips_fn(x_, y_).mean().item())

@torch.no_grad()
def sample_sr_no_noise(model, lr):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    noise_img = torch.zeros(
        B, cfg.noise_image_channels, H, W,
        device=lr.device, dtype=lr.dtype
    )
    noise_vec = torch.zeros(
        B, cfg.noise_embed_dim,
        device=lr.device, dtype=lr.dtype
    )

    pred, up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return pred.clamp(0, 1), up.clamp(0, 1)

@torch.no_grad()
def sample_sr_with_noise_seed(model, lr, cfg, noise_seed):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    gen = torch.Generator(device=lr.device)
    gen.manual_seed(int(noise_seed))

    noise_img = torch.randn(
        B, cfg.noise_image_channels, H, W,
        device=lr.device, dtype=lr.dtype, generator=gen
    )
    noise_vec = torch.randn(
        B, cfg.noise_embed_dim,
        device=lr.device, dtype=lr.dtype, generator=gen
    )

    pred, up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return pred.clamp(0, 1), up.clamp(0, 1)

def print_running_stats(df_fidelity, df_pairs, processed, total):
    msg = [f"[{processed}/{total}]"]

    if len(df_fidelity) > 0:
        msg.append(
            "fidelity:"
            f" psnr={df_fidelity['psnr_pred'].mean():.4f},"
            f" lpips={df_fidelity['lpips_pred'].mean():.6f},"
            f" bic_psnr={df_fidelity['psnr_bicubic'].mean():.4f},"
            f" bic_lpips={df_fidelity['lpips_bicubic'].mean():.6f}"
        )

    if len(df_pairs) > 0:
        msg.append(
            "pairs:"
            f" pair_lpips={df_pairs['pair_lpips'].mean():.6f},"
            f" pair_l1={df_pairs['pair_l1'].mean():.6f},"
            f" pair_residual_l1={df_pairs['pair_residual_l1'].mean():.6f},"
            f" pair_down_l1={df_pairs['pair_down_l1'].mean():.6f}"
        )

    print(" | ".join(msg))

# =========================
# eval
# =========================
rows_fidelity = []
rows_pairs = []

for ds_idx in range(len(dataset)):
    hr, lr = dataset[ds_idx]
    hr = hr.to(device)
    lr = lr.to(device)

    hr_in = hr.unsqueeze(0)
    lr_in = lr.unsqueeze(0)

    # -------------------------
    # 1) deterministic fidelity
    # -------------------------
    if USE_ZERO_NOISE_FOR_FIDELITY:
        pred_det, bic = sample_sr_no_noise(model, lr_in)
    else:
        pred_det, bic = sample_sr_with_noise_seed(model, lr_in, cfg, noise_seed=PAIR_NOISE_SEEDS[0])

    psnr_pred = calc_psnr_sr(
        pred_det, hr_in,
        shave=cfg.shave_border,
        use_y=cfg.eval_on_y_channel
    )
    psnr_bic = calc_psnr_sr(
        bic, hr_in,
        shave=cfg.shave_border,
        use_y=cfg.eval_on_y_channel
    )

    lpips_pred = calc_lpips01(pred_det, hr_in)
    lpips_bic = calc_lpips01(bic, hr_in)

    rows_fidelity.append({
        "checkpoint": Path(CHECKPOINT_PATH).name,
        "dataset_index": ds_idx,
        "psnr_pred": psnr_pred,
        "psnr_bicubic": psnr_bic,
        "lpips_pred": lpips_pred,
        "lpips_bicubic": lpips_bic,
    })

    # -------------------------
    # 2) stochastic pair metrics
    # -------------------------
    preds = []
    for noise_seed in PAIR_NOISE_SEEDS:
        pred_s, _ = sample_sr_with_noise_seed(model, lr_in, cfg, noise_seed=noise_seed)
        preds.append((int(noise_seed), pred_s))

    for (seed_a, pred_a), (seed_b, pred_b) in itertools.combinations(preds, 2):
        pair_lpips = calc_lpips01(pred_a, pred_b)
        pair_l1 = float(F.l1_loss(pred_a, pred_b).item())

        res_a = pred_a - bic
        res_b = pred_b - bic
        pair_residual_l1 = float(F.l1_loss(res_a, res_b).item())

        down_a = F.interpolate(pred_a, size=lr_in.shape[-2:], mode="bicubic", align_corners=False)
        down_b = F.interpolate(pred_b, size=lr_in.shape[-2:], mode="bicubic", align_corners=False)
        pair_down_l1 = float(F.l1_loss(down_a, down_b).item())

        rows_pairs.append({
            "checkpoint": Path(CHECKPOINT_PATH).name,
            "dataset_index": ds_idx,
            "seed_a": int(seed_a),
            "seed_b": int(seed_b),
            "pair_lpips": pair_lpips,
            "pair_l1": pair_l1,
            "pair_residual_l1": pair_residual_l1,
            "pair_down_l1": pair_down_l1,
        })

    # -------------------------
    # progress prints
    # -------------------------
    processed = ds_idx + 1
    if processed % PROGRESS_EVERY == 0 or processed == len(dataset):
        df_fid_tmp = pd.DataFrame(rows_fidelity)
        df_pair_tmp = pd.DataFrame(rows_pairs)
        print_running_stats(df_fid_tmp, df_pair_tmp, processed, len(dataset))

# =========================
# DataFrames
# =========================
df_fidelity = pd.DataFrame(rows_fidelity)
df_pairs = pd.DataFrame(rows_pairs)

# =========================
# summary per image
# =========================
summary_pairs_per_image = (
    df_pairs
    .groupby(["dataset_index"])
    .agg(
        pair_lpips_mean=("pair_lpips", "mean"),
        pair_lpips_std=("pair_lpips", "std"),
        pair_l1_mean=("pair_l1", "mean"),
        pair_l1_std=("pair_l1", "std"),
        pair_residual_l1_mean=("pair_residual_l1", "mean"),
        pair_residual_l1_std=("pair_residual_l1", "std"),
        pair_down_l1_mean=("pair_down_l1", "mean"),
        pair_down_l1_std=("pair_down_l1", "std"),
    )
    .reset_index()
)

summary_per_image = df_fidelity.merge(summary_pairs_per_image, on="dataset_index", how="left")

# =========================
# global summary
# =========================
summary = pd.DataFrame([{
    "checkpoint": Path(CHECKPOINT_PATH).name,
    "num_images": int(len(df_fidelity)),
    "pair_seeds": str(PAIR_NOISE_SEEDS),
    "num_pairs_per_image": int(len(PAIR_NOISE_SEEDS) * (len(PAIR_NOISE_SEEDS)-1) // 2),

    "mean_psnr_pred": float(df_fidelity["psnr_pred"].mean()),
    "mean_psnr_bicubic": float(df_fidelity["psnr_bicubic"].mean()),
    "mean_lpips_pred": float(df_fidelity["lpips_pred"].mean()),
    "mean_lpips_bicubic": float(df_fidelity["lpips_bicubic"].mean()),

    "mean_pair_lpips": float(df_pairs["pair_lpips"].mean()),
    "std_pair_lpips": float(df_pairs["pair_lpips"].std()),
    "mean_pair_l1": float(df_pairs["pair_l1"].mean()),
    "std_pair_l1": float(df_pairs["pair_l1"].std()),
    "mean_pair_residual_l1": float(df_pairs["pair_residual_l1"].mean()),
    "std_pair_residual_l1": float(df_pairs["pair_residual_l1"].std()),
    "mean_pair_down_l1": float(df_pairs["pair_down_l1"].mean()),
    "std_pair_down_l1": float(df_pairs["pair_down_l1"].std()),
}])

# =========================
# save
# =========================
if SAVE_CSV:
    fidelity_csv = os.path.join(OUTPUT_DIR, "full_val_fidelity_no_noise.csv")
    pairs_csv = os.path.join(OUTPUT_DIR, "full_val_pair_metrics.csv")
    per_image_csv = os.path.join(OUTPUT_DIR, "full_val_summary_per_image.csv")
    summary_csv = os.path.join(OUTPUT_DIR, "full_val_summary.csv")

    df_fidelity.to_csv(fidelity_csv, index=False)
    df_pairs.to_csv(pairs_csv, index=False)
    summary_per_image.to_csv(per_image_csv, index=False)
    summary.to_csv(summary_csv, index=False)

if SAVE_JSON:
    summary_json = os.path.join(OUTPUT_DIR, "full_val_summary.json")
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary.iloc[0].to_dict(), f, ensure_ascii=False, indent=2)

# =========================
# print
# =========================
print("\n=== PER-IMAGE FIDELITY ===")
display(df_fidelity)

print("\n=== PER-PAIR METRICS ===")
display(df_pairs)

print("\n=== SUMMARY PER IMAGE ===")
display(summary_per_image)

print("\n=== GLOBAL SUMMARY ===")
display(summary)

print("\nSaved to:", OUTPUT_DIR)


Dataset size: 100
Checkpoint: /kaggle/input/datasets/ksenia1589/epoch-40/epoch_040.pt
Pair noise seeds: [0, 1, 2] -> 3 pairs/image
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 171MB/s]  


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[10/100] | fidelity: psnr=27.4368, lpips=0.350240, bic_psnr=27.7598, bic_lpips=0.345747 | pairs: pair_lpips=0.029174, pair_l1=0.002422, pair_residual_l1=0.002422, pair_down_l1=0.000242
[20/100] | fidelity: psnr=27.4324, lpips=0.347469, bic_psnr=27.7653, bic_lpips=0.343117 | pairs: pair_lpips=0.032926, pair_l1=0.002471, pair_residual_l1=0.002471, pair_down_l1=0.000257
[30/100] | fidelity: psnr=26.5886, lpips=0.351031, bic_psnr=26.7808, bic_lpips=0.351744 | pairs: pair_lpips=0.031914, pair_l1=0.002485, pair_residual_l1=0.002485, pair_down_l1=0.000301
[40/100] | fidelity: psnr=26.6555, lpips=0.353700, bic_psnr=26.8912, bic_lpips=0.352491 | pairs: pair_lpips=0.031602, pair_l1=0.002457, pair_residual_l1=0.002457, pair_down_l1=0.000295
[50/100] | fidelity: psnr=26.7535, lpips=0.351018, bic_psnr=26.9598, bic_lpips=0.348641 | pairs: pair_lpips=0.033719, pair_l1=0.002451, pair_residual_l1=0.002451, pair_down_

...


=== SUMMARY PER IMAGE ===


,checkpoint,dataset_index,psnr_pred,psnr_bicubic,lpips_pred,lpips_bicubic,pair_lpips_mean,pair_lpips_std,pair_l1_mean,pair_l1_std,pair_residual_l1_mean,pair_residual_l1_std,pair_down_l1_mean,pair_down_l1_std
0,epoch_040.pt,0,27.225840,27.784136,0.389818,0.329026,0.018807,0.001777,0.002322,0.000191,0.002322,0.000191,0.000246,0.000012
1,epoch_040.pt,1,32.366703,33.764965,0.205294,0.213919,0.017881,0.000313,0.002617,0.000041,0.002617,0.000041,0.000172,0.000008
2,epoch_040.pt,2,34.189896,33.956917,0.255108,0.228932,0.056772,0.000180,0.003068,0.000057,0.003068,0.000057,0.000196,0.000012
3,epoch_040.pt,3,25.801735,25.704222,0.347393,0.352412,0.018075,0.000833,0.002432,0.000144,0.002432,0.000144,0.000281,0.000012
4,epoch_040.pt,4,27.150452,27.614622,0.425196,0.424505,0.019842,0.001696,0.002302,0.000144,0.002302,0.000144,0.000264,0.000031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,epoch_040.pt,95,34.402008,33.906025,0.320094,0.349816,0.063953,0.003471,0.001997,0.000091,0.001997,0.000091,0.000154,0.000009
96,epoch_040.pt,96,20.660975,20.770344,0.354974,0.404800,0.003981,0.000333,0.002299,0.000114,0.002299,0.000114,0.000497,0.000025
97,epoch_040.pt,97,31.679466,31.961782,0.301312,0.335687,0.044468,0.001371,0.002008,0.000084,0.002008,0.000084,0.000152,0.000009
98,epoch_040.pt,98,27.399733,26.362011,0.295322,0.262098,0.027741,0.000112,0.002247,0.000050,0.002247,0.000050,0.000285,0.000013



=== GLOBAL SUMMARY ===


,checkpoint,num_images,pair_seeds,num_pairs_per_image,mean_psnr_pred,mean_psnr_bicubic,mean_lpips_pred,mean_lpips_bicubic,mean_pair_lpips,std_pair_lpips,mean_pair_l1,std_pair_l1,mean_pair_residual_l1,std_pair_residual_l1,mean_pair_down_l1,std_pair_down_l1
0,epoch_040.pt,100,"[0, 1, 2]",3,26.695993,26.907455,0.361187,0.360611,0.033558,0.019377,0.00241,0.000275,0.00241,0.000275,0.000283,0.000134



Saved to: /kaggle/working/sr_40


In [3]:
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import lpips

# =========================
# настройки
# =========================
CHECKPOINT_PATH = "/kaggle/input/datasets/ksenia1589/08rep-lpips/best_lpips.pt"
OUTPUT_DIR = "/kaggle/working/sr_lpips"

USE_VAL_DATASET = True
SAVE_CSV = True
SAVE_JSON = True

# deterministic метрики к GT
USE_ZERO_NOISE_FOR_FIDELITY = True

# stochastic seeds только для pair-метрик
PAIR_NOISE_SEEDS = [0, 1, 2]   # 3 seeds -> 3 pair'а на изображение
PROGRESS_EVERY = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = cfg.device if torch.cuda.is_available() else "cpu"

# =========================
# dataset
# =========================
dataset = DIV2KPairDataset(
    hr_dir=cfg.val_hr_dir if USE_VAL_DATASET else cfg.train_hr_dir,
    lr_dir=cfg.val_lr_dir if USE_VAL_DATASET else cfg.train_lr_dir,
    scale=cfg.scale,
    patch_size=None,
    training=False,
)

print(f"Dataset size: {len(dataset)}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Pair noise seeds: {PAIR_NOISE_SEEDS} -> {len(PAIR_NOISE_SEEDS) * (len(PAIR_NOISE_SEEDS)-1) // 2} pairs/image")

# =========================
# model + checkpoint
# =========================
model = NoiseConditionalResidualUNetSR(
    in_channels=cfg.in_channels,
    noise_image_channels=cfg.noise_image_channels,
    base=cfg.unet_base,
    channel_mult=cfg.unet_channel_mult,
    num_blocks=cfg.unet_num_blocks,
    noise_embed_dim=cfg.noise_embed_dim,
    residual_out_scale=cfg.residual_out_scale,
    scale=cfg.scale,
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
state = ckpt["ema_state"] if "ema_state" in ckpt else ckpt["model_state"]
model.load_state_dict(state, strict=True)
model.eval()

# =========================
# LPIPS
# =========================
lpips_fn = lpips.LPIPS(net="vgg").to(device).eval()
for p in lpips_fn.parameters():
    p.requires_grad = False

# =========================
# helpers
# =========================
@torch.no_grad()
def calc_lpips01(x, y):
    x_ = x.clamp(0, 1) * 2 - 1
    y_ = y.clamp(0, 1) * 2 - 1
    return float(lpips_fn(x_, y_).mean().item())

@torch.no_grad()
def sample_sr_no_noise(model, lr):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    noise_img = torch.zeros(
        B, cfg.noise_image_channels, H, W,
        device=lr.device, dtype=lr.dtype
    )
    noise_vec = torch.zeros(
        B, cfg.noise_embed_dim,
        device=lr.device, dtype=lr.dtype
    )

    pred, up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return pred.clamp(0, 1), up.clamp(0, 1)

@torch.no_grad()
def sample_sr_with_noise_seed(model, lr, cfg, noise_seed):
    B = lr.shape[0]
    H = lr.shape[-2] * cfg.scale
    W = lr.shape[-1] * cfg.scale

    gen = torch.Generator(device=lr.device)
    gen.manual_seed(int(noise_seed))

    noise_img = torch.randn(
        B, cfg.noise_image_channels, H, W,
        device=lr.device, dtype=lr.dtype, generator=gen
    )
    noise_vec = torch.randn(
        B, cfg.noise_embed_dim,
        device=lr.device, dtype=lr.dtype, generator=gen
    )

    pred, up = model(lr, noise_img=noise_img, noise_vec=noise_vec)
    return pred.clamp(0, 1), up.clamp(0, 1)

def print_running_stats(df_fidelity, df_pairs, processed, total):
    msg = [f"[{processed}/{total}]"]

    if len(df_fidelity) > 0:
        msg.append(
            "fidelity:"
            f" psnr={df_fidelity['psnr_pred'].mean():.4f},"
            f" lpips={df_fidelity['lpips_pred'].mean():.6f},"
            f" bic_psnr={df_fidelity['psnr_bicubic'].mean():.4f},"
            f" bic_lpips={df_fidelity['lpips_bicubic'].mean():.6f}"
        )

    if len(df_pairs) > 0:
        msg.append(
            "pairs:"
            f" pair_lpips={df_pairs['pair_lpips'].mean():.6f},"
            f" pair_l1={df_pairs['pair_l1'].mean():.6f},"
            f" pair_residual_l1={df_pairs['pair_residual_l1'].mean():.6f},"
            f" pair_down_l1={df_pairs['pair_down_l1'].mean():.6f}"
        )

    print(" | ".join(msg))

# =========================
# eval
# =========================
rows_fidelity = []
rows_pairs = []

for ds_idx in range(len(dataset)):
    hr, lr = dataset[ds_idx]
    hr = hr.to(device)
    lr = lr.to(device)

    hr_in = hr.unsqueeze(0)
    lr_in = lr.unsqueeze(0)

    # -------------------------
    # 1) deterministic fidelity
    # -------------------------
    if USE_ZERO_NOISE_FOR_FIDELITY:
        pred_det, bic = sample_sr_no_noise(model, lr_in)
    else:
        pred_det, bic = sample_sr_with_noise_seed(model, lr_in, cfg, noise_seed=PAIR_NOISE_SEEDS[0])

    psnr_pred = calc_psnr_sr(
        pred_det, hr_in,
        shave=cfg.shave_border,
        use_y=cfg.eval_on_y_channel
    )
    psnr_bic = calc_psnr_sr(
        bic, hr_in,
        shave=cfg.shave_border,
        use_y=cfg.eval_on_y_channel
    )

    lpips_pred = calc_lpips01(pred_det, hr_in)
    lpips_bic = calc_lpips01(bic, hr_in)

    rows_fidelity.append({
        "checkpoint": Path(CHECKPOINT_PATH).name,
        "dataset_index": ds_idx,
        "psnr_pred": psnr_pred,
        "psnr_bicubic": psnr_bic,
        "lpips_pred": lpips_pred,
        "lpips_bicubic": lpips_bic,
    })

    # -------------------------
    # 2) stochastic pair metrics
    # -------------------------
    preds = []
    for noise_seed in PAIR_NOISE_SEEDS:
        pred_s, _ = sample_sr_with_noise_seed(model, lr_in, cfg, noise_seed=noise_seed)
        preds.append((int(noise_seed), pred_s))

    for (seed_a, pred_a), (seed_b, pred_b) in itertools.combinations(preds, 2):
        pair_lpips = calc_lpips01(pred_a, pred_b)
        pair_l1 = float(F.l1_loss(pred_a, pred_b).item())

        res_a = pred_a - bic
        res_b = pred_b - bic
        pair_residual_l1 = float(F.l1_loss(res_a, res_b).item())

        down_a = F.interpolate(pred_a, size=lr_in.shape[-2:], mode="bicubic", align_corners=False)
        down_b = F.interpolate(pred_b, size=lr_in.shape[-2:], mode="bicubic", align_corners=False)
        pair_down_l1 = float(F.l1_loss(down_a, down_b).item())

        rows_pairs.append({
            "checkpoint": Path(CHECKPOINT_PATH).name,
            "dataset_index": ds_idx,
            "seed_a": int(seed_a),
            "seed_b": int(seed_b),
            "pair_lpips": pair_lpips,
            "pair_l1": pair_l1,
            "pair_residual_l1": pair_residual_l1,
            "pair_down_l1": pair_down_l1,
        })

    # -------------------------
    # progress prints
    # -------------------------
    processed = ds_idx + 1
    if processed % PROGRESS_EVERY == 0 or processed == len(dataset):
        df_fid_tmp = pd.DataFrame(rows_fidelity)
        df_pair_tmp = pd.DataFrame(rows_pairs)
        print_running_stats(df_fid_tmp, df_pair_tmp, processed, len(dataset))

# =========================
# DataFrames
# =========================
df_fidelity = pd.DataFrame(rows_fidelity)
df_pairs = pd.DataFrame(rows_pairs)

# =========================
# summary per image
# =========================
summary_pairs_per_image = (
    df_pairs
    .groupby(["dataset_index"])
    .agg(
        pair_lpips_mean=("pair_lpips", "mean"),
        pair_lpips_std=("pair_lpips", "std"),
        pair_l1_mean=("pair_l1", "mean"),
        pair_l1_std=("pair_l1", "std"),
        pair_residual_l1_mean=("pair_residual_l1", "mean"),
        pair_residual_l1_std=("pair_residual_l1", "std"),
        pair_down_l1_mean=("pair_down_l1", "mean"),
        pair_down_l1_std=("pair_down_l1", "std"),
    )
    .reset_index()
)

summary_per_image = df_fidelity.merge(summary_pairs_per_image, on="dataset_index", how="left")

# =========================
# global summary
# =========================
summary = pd.DataFrame([{
    "checkpoint": Path(CHECKPOINT_PATH).name,
    "num_images": int(len(df_fidelity)),
    "pair_seeds": str(PAIR_NOISE_SEEDS),
    "num_pairs_per_image": int(len(PAIR_NOISE_SEEDS) * (len(PAIR_NOISE_SEEDS)-1) // 2),

    "mean_psnr_pred": float(df_fidelity["psnr_pred"].mean()),
    "mean_psnr_bicubic": float(df_fidelity["psnr_bicubic"].mean()),
    "mean_lpips_pred": float(df_fidelity["lpips_pred"].mean()),
    "mean_lpips_bicubic": float(df_fidelity["lpips_bicubic"].mean()),

    "mean_pair_lpips": float(df_pairs["pair_lpips"].mean()),
    "std_pair_lpips": float(df_pairs["pair_lpips"].std()),
    "mean_pair_l1": float(df_pairs["pair_l1"].mean()),
    "std_pair_l1": float(df_pairs["pair_l1"].std()),
    "mean_pair_residual_l1": float(df_pairs["pair_residual_l1"].mean()),
    "std_pair_residual_l1": float(df_pairs["pair_residual_l1"].std()),
    "mean_pair_down_l1": float(df_pairs["pair_down_l1"].mean()),
    "std_pair_down_l1": float(df_pairs["pair_down_l1"].std()),
}])

# =========================
# save
# =========================
if SAVE_CSV:
    fidelity_csv = os.path.join(OUTPUT_DIR, "full_val_fidelity_no_noise.csv")
    pairs_csv = os.path.join(OUTPUT_DIR, "full_val_pair_metrics.csv")
    per_image_csv = os.path.join(OUTPUT_DIR, "full_val_summary_per_image.csv")
    summary_csv = os.path.join(OUTPUT_DIR, "full_val_summary.csv")

    df_fidelity.to_csv(fidelity_csv, index=False)
    df_pairs.to_csv(pairs_csv, index=False)
    summary_per_image.to_csv(per_image_csv, index=False)
    summary.to_csv(summary_csv, index=False)

if SAVE_JSON:
    summary_json = os.path.join(OUTPUT_DIR, "full_val_summary.json")
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary.iloc[0].to_dict(), f, ensure_ascii=False, indent=2)

# =========================
# print
# =========================
print("\n=== PER-IMAGE FIDELITY ===")
display(df_fidelity)

print("\n=== PER-PAIR METRICS ===")
display(df_pairs)

print("\n=== SUMMARY PER IMAGE ===")
display(summary_per_image)

print("\n=== GLOBAL SUMMARY ===")
display(summary)

print("\nSaved to:", OUTPUT_DIR)


Dataset size: 100
Checkpoint: /kaggle/input/datasets/ksenia1589/08rep-lpips/best_lpips.pt
Pair noise seeds: [0, 1, 2] -> 3 pairs/image
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 194MB/s]  


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
[10/100] | fidelity: psnr=27.8757, lpips=0.312932, bic_psnr=27.7598, bic_lpips=0.345747 | pairs: pair_lpips=0.016074, pair_l1=0.001135, pair_residual_l1=0.001135, pair_down_l1=0.000127
[20/100] | fidelity: psnr=27.8849, lpips=0.314303, bic_psnr=27.7653, bic_lpips=0.343117 | pairs: pair_lpips=0.018868, pair_l1=0.001174, pair_residual_l1=0.001174, pair_down_l1=0.000129
[30/100] | fidelity: psnr=26.9041, lpips=0.323316, bic_psnr=26.7808, bic_lpips=0.351744 | pairs: pair_lpips=0.017923, pair_l1=0.001142, pair_residual_l1=0.001142, pair_down_l1=0.000130
[40/100] | fidelity: psnr=27.0125, lpips=0.327179, bic_psnr=26.8912, bic_lpips=0.352491 | pairs: pair_lpips=0.017983, pair_l1=0.001132, pair_residual_l1=0.001132, pair_down_l1=0.000132
[50/100] | fidelity: psnr=27.0861, lpips=0.323662, bic_psnr=26.9598, bic_lpips=0.348641 | pairs: pair_lpips=0.018621, pair_l1=0.001122, pair_residual_l1=0.001122, pair_down_

...


=== SUMMARY PER IMAGE ===


,checkpoint,dataset_index,psnr_pred,psnr_bicubic,lpips_pred,lpips_bicubic,pair_lpips_mean,pair_lpips_std,pair_l1_mean,pair_l1_std,pair_residual_l1_mean,pair_residual_l1_std,pair_down_l1_mean,pair_down_l1_std
0,best_lpips.pt,0,27.908480,27.784136,0.312047,0.329026,0.008747,0.000873,0.000990,0.000040,0.000990,0.000040,0.000120,0.000008
1,best_lpips.pt,1,33.871902,33.764965,0.188029,0.213919,0.008118,0.000177,0.001319,0.000015,0.001319,0.000015,0.000132,0.000004
2,best_lpips.pt,2,34.343704,33.956917,0.161918,0.228932,0.033293,0.001061,0.001742,0.000023,0.001742,0.000023,0.000156,0.000002
3,best_lpips.pt,3,25.817513,25.704222,0.336885,0.352412,0.008592,0.000474,0.001033,0.000009,0.001033,0.000009,0.000116,0.000002
4,best_lpips.pt,4,27.686306,27.614622,0.391661,0.424505,0.008970,0.000600,0.000895,0.000035,0.000895,0.000035,0.000119,0.000006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,best_lpips.pt,95,34.370476,33.906025,0.349319,0.349816,0.049087,0.006994,0.001083,0.000060,0.001083,0.000060,0.000110,0.000007
96,best_lpips.pt,96,20.832531,20.770344,0.401960,0.404800,0.001046,0.000400,0.000688,0.000110,0.000688,0.000110,0.000151,0.000020
97,best_lpips.pt,97,32.157997,31.961782,0.344899,0.335687,0.035271,0.002728,0.001169,0.000044,0.001169,0.000044,0.000116,0.000007
98,best_lpips.pt,98,26.635103,26.362011,0.244102,0.262098,0.010540,0.000404,0.001072,0.000015,0.001072,0.000015,0.000116,0.000004



=== GLOBAL SUMMARY ===


,checkpoint,num_images,pair_seeds,num_pairs_per_image,mean_psnr_pred,mean_psnr_bicubic,mean_lpips_pred,mean_lpips_bicubic,mean_pair_lpips,std_pair_lpips,mean_pair_l1,std_pair_l1,mean_pair_residual_l1,std_pair_residual_l1,mean_pair_down_l1,std_pair_down_l1
0,best_lpips.pt,100,"[0, 1, 2]",3,27.029173,26.907455,0.336815,0.360611,0.019149,0.013509,0.001121,0.000231,0.001121,0.000231,0.00013,0.000016



Saved to: /kaggle/working/sr_lpips
